# mainscript — dblink 직접 실행용 노트북

논문 1편을 넣으면 **Methods 섹션 추출 → 단어 추출 → 외부 DB 매칭** 까지 한 번에 돌려봅니다.

## 사용법
1. 아래 **[입력]** 셀의 `inputpaper` 를 분석할 논문 파일 경로로 바꾼다.
2. 위에서부터 셀을 순서대로 실행한다 (또는 메뉴 `Run > Run All Cells`).
3. 맨 아래 **[출력]** 표에서 매칭 결과를 확인한다.

> ⚠️ 이 노트북은 `01_dblink` 폴더 안에서 실행해야 합니다 (같은 폴더의 `dblink.py` 등을 불러오기 때문).
> `.txt` 와 `.pdf` 둘 다 됩니다. `.pdf` 는 `pypdf` 설치가 필요합니다: `pip install pypdf`

## [입력] 분석할 논문 파일 경로

In [ ]:
# =========================================================
# [입력] 여기에 분석할 논문 파일 경로를 적으세요 (.txt 또는 .pdf)
# =========================================================
#
#  - 이 노트북과 같은 01_dblink 폴더에 파일을 두면 '파일이름'만 쓰면 됩니다.
#      예) inputpaper = "kim2022.txt"
#  - 다른 폴더에 있으면 전체 경로를 쓰세요.
#      예) inputpaper = "/Users/hyeree/Downloads/kim2022.pdf"

inputpaper = "여기에_논문파일_경로.txt"   # ← 이 부분을 본인 파일로 바꾸세요

## 1단계 — 필요한 함수 불러오기 + 논문 파일 읽기

같은 폴더의 4개 모듈에서 필요한 함수만 가져옵니다. 그다음 파일을 읽어 전체 텍스트(`full_text`)를 만듭니다.

In [ ]:
import os
from extract_methods   import read_pdf_text, find_methods   # 논문 전체 -> Methods 섹션
from extract_candidates import extract_candidates            # Methods -> 후보 단어
from pipeline          import build_acronym_map, classify_by_db  # 약어사전 / DB 매칭
from dblink            import canonical_key                  # 표현 정규화 키

# --- 파일 읽기 (.pdf 면 pypdf 로 텍스트 추출, .txt 면 그대로 읽기) ---
if not os.path.exists(inputpaper):
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {inputpaper}\n-> [입력] 셀의 경로를 확인하세요.")

if inputpaper.lower().endswith(".pdf"):
    full_text = read_pdf_text(inputpaper)      # pypdf 필요: pip install pypdf
else:
    with open(inputpaper, encoding="utf-8") as f:
        full_text = f.read()

print(f"[입력 확인] 파일: {inputpaper}")
print(f"[입력 확인] 논문 전체 글자 수: {len(full_text):,}")

## 2단계 — Methods(실험) 섹션 추출 + 약어사전 만들기

- `find_methods` : 'Methods / Experimental Section …' 제목부터 다음 섹션 직전까지를 잘라냅니다.
- `build_acronym_map` : 본문 전체에서 `풀네임 (약어)` 형태를 찾아 약어사전을 만듭니다.
  (예: `benzylphosphonic acid (BPA)` → `BPA: benzylphosphonic acid`). 매칭할 때 약어를 풀네임으로 바꿔 질의하는 데 씁니다.

In [ ]:
acronym_map  = build_acronym_map(full_text)   # {약어: 풀네임} 사전 (본문 전체 기준)
methods_text = find_methods(full_text)        # Methods 섹션 텍스트 (못 찾으면 None)

if methods_text is None:
    print("⚠️ Methods 섹션을 찾지 못했습니다.")
    print("   - 논문에 'Methods' 또는 'Experimental Section' 같은 제목이 '단독 줄'로 있는지 확인하세요.")
    print("   - PDF라면 제목 줄이 깨졌을 수 있습니다.")
else:
    print(f"[2단계 결과] Methods 섹션 글자 수: {len(methods_text):,}")
    print(f"[2단계 결과] 약어사전 등록 개수: {len(acronym_map)}개  ->  {acronym_map}")
    print("\n----- Methods 앞부분 미리보기 (500자) -----")
    print(methods_text[:500])

## 3단계 — 단어(후보) 추출

Methods 텍스트에서 **대문자로 시작하는 표현 / 전부 대문자 약어 / 화학식**을 후보로 뽑습니다.
재현율 위주의 거친 후보이며, 진짜/가짜 판별은 다음 매칭 단계에서 걸러집니다.

In [ ]:
candidates = extract_candidates(methods_text)

print(f"[3단계 결과] 후보 단어/표현: {len(candidates)}개\n")
print(candidates)

## 4단계 — 외부 DB 매칭 (DB 역분류)

후보 하나하나를 외부 DB에 질의해 **카테고리 + 정규화 ID**를 붙입니다.
- PubChem 매칭 → `ingredient` (화학물질)
- CHMO 매칭 → `method` (분석법)
- OBI 매칭 → `equipment` (장비)
- 약어는 약어사전으로 풀네임 치환 후 질의, 사전에 없는 맨 약어는 건너뜀.

> 🌐 이 단계는 **인터넷**이 필요하고, 후보 수에 따라 시간이 조금 걸립니다 (후보당 약 0.5~2초).
> 한 번 질의한 건 `_dblink_cache.json` 에 저장되어 다음부터는 빨라집니다.

In [ ]:
results = []
for surface in candidates:
    hit = classify_by_db(surface, acronym_map)   # 매칭되면 dict, 아니면 None
    if hit:
        hit["surface"]       = surface
        hit["canonical_key"] = canonical_key(surface)
        results.append(hit)

print(f"[4단계 결과] 매칭 성공: {len(results)}개 / 후보 {len(candidates)}개")

## [출력] 매칭 결과 보기

매칭된 항목을 표로 정리해서 보여줍니다.

| 열 | 뜻 |
|---|---|
| surface | 논문에 실제로 나온 표현 |
| category | 분류 (ingredient / method / equipment) |
| DB | 매칭된 외부 데이터베이스 |
| ID | 그 DB에서의 정규화 ID |
| query | DB에 실제로 던진 질의어 (약어는 풀네임으로 치환됨) |

In [ ]:
rows = []
for r in results:
    nid = r["normalized_ids"][0]              # 대표 ID 하나
    rows.append({
        "surface":  r["surface"],
        "category": r["category"],
        "DB":       nid["db"],
        "ID":       nid["id"],
        "query":    r["query"],
    })

if not rows:
    print("매칭된 항목이 없습니다. (후보가 DB에 없거나, Methods를 못 찾았을 수 있어요)")
else:
    try:
        import pandas as pd
        df = pd.DataFrame(rows)
        display(df)                           # Jupyter 표로 보기 좋게 출력
    except ImportError:
        # pandas 가 없으면 줄글로 출력
        for row in rows:
            print(row)